# Transcript Examples — Loading and Rendering Game Transcripts

This notebook demonstrates how to load saved game transcripts and render
board snapshots and HTML reports.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt

from scripts.render_transcripts import (
    load_transcripts,
    render_board_png,
    render_tic_tac_toe_board,
    render_transcript_html,
    select_bad_game,
    select_good_game,
)

## 1. Load all transcripts from a directory

In [ ]:
transcripts_dir = "transcripts/"
transcripts = load_transcripts(transcripts_dir)
print(f"Loaded {len(transcripts)} transcripts")

# Show a summary table
for t in transcripts[:5]:
    print(f"  {t['match_id'][:8]}…: {t['agent_a']:>25} vs {t['agent_b']:<25} winner={t['winner']}")

## 2. Auto-select good and bad games

In [ ]:
good = select_good_game(transcripts)
bad = select_bad_game(transcripts)

print(f"Good game: {good['match_id']}")
print(f"  {good['agent_a']} vs {good['agent_b']} → winner={good['winner']}, moves={good['num_moves']}")
print()
print(f"Bad game:  {bad['match_id']}")
print(f"  {bad['agent_a']} vs {bad['agent_b']} → winner={bad['winner']}, retries={bad['invalid_move_retries']}")

## 3. Inspect individual moves

In [ ]:
# Show moves from the good game
for entry in good['entries']:
    llm_tag = ""
    if entry.get('llm_prompt'):
        llm_tag = " (LLM)"
    print(f"  Move {entry['move_num']}: Player {entry['player']} ({entry['agent_name']}{llm_tag}) → action {entry['action']}")
    print(f"    Board:\n{entry['board_str']}")
    print()

## 4. Render board snapshots

In [ ]:
# Render the good game's final board to PNG
render_board_png(good, "output/notebook_good_game.png")
print("Saved: output/notebook_good_game.png")

# Display inline
from IPython.display import Image, display
display(Image(filename="output/notebook_good_game.png"))

## 5. Generate HTML reports

In [ ]:
# Generate HTML for both games
render_transcript_html(good, "output/notebook_good_game.html", label="Good Game")
render_transcript_html(bad, "output/notebook_bad_game.html", label="Bad Game")
print("Saved: output/notebook_good_game.html")
print("Saved: output/notebook_bad_game.html")

## 6. Load and inspect a specific transcript by match_id

In [ ]:
# Pick any transcript by ID
match_id = transcripts[0]['match_id']
t = json.loads(Path(f"transcripts/{match_id}.json").read_text())
print(json.dumps({k: v for k, v in t.items() if k != 'entries'}, indent=2))
print(f"\nMoves played: {len(t['entries'])}")